![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 45 -- Lab 1: Hidden Sketch

A company hides its logo, a small black line drawing we call the sketch, inside photos: turned, mirrored, made bigger or smaller, at most once per photo. Some photos have no sketch at all. You write the program that finds the box around the sketch, or says that there is none. Your file is scored with the mean IoU of the boxes over the test photos. Higher is better.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Photos as PNG files. The training photos come with the box of their sketch; the test photos do not, and only the scorer holds their boxes. The setup cell leaves in `DATA_DIR`: `train.csv` (`id`, `image`, `has_sketch`, `box`), `test.csv` (`id`, `image`), `sample_submission.csv` (the format of the file you hand in) and the folders `train/` and `test/`. **Use a GPU:** in Colab, Runtime, Change runtime type, T4 GPU, before you run the first cell.

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab ultralytics
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="cv-02", student_id=STUDENT_ID)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print("libraries ready")


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/hidden-sketch")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# The task

**The story.** A company hides its logo in photos: a small black line drawing, which we call **the sketch**. The
sketch may be turned, mirrored, made bigger or smaller, and pasted anywhere in the photo, at most once. Some photos
have no sketch at all. Your program must find the sketch, or say that there is none.

**One data point** is one photo and its box:

- the **photo**, `train/<id>.png`: a colour picture (open it with `Image.open(...).convert("RGB")`);
- the **box** of its sketch: four whole numbers `x1 y1 x2 y2`, in pixels of that photo. `x1 y1` is the top-left
  corner of the box and `x2 y2` its bottom-right corner; x counts from the left edge of the photo to the right, y
  from the top edge down. The box is as small as possible: every pixel of the sketch lies inside it. A photo without
  a sketch has the box `0 0 0 0`. The column `has_sketch` says the same in one number: 1 if there is a sketch, 0 if
  not.

**The files.** The first rows of `train.csv` and `test.csv`:

| id | image | has_sketch | box |
|---|---|---|---|
| 0 | train/0.png | 1 | 204 321 275 410 |
| 1 | train/1.png | 1 | 77 106 172 192 |

The first row says: in photo 0 the sketch lies between x = 204 and x = 275, and between y = 321 and y = 410.

| id | image |
|---|---|
| 1300 | test/1300.png |
| 1301 | test/1301.png |

**The answer** for a test photo is its box, written the same way, or `0 0 0 0` when you think there is no sketch.
The task had three parts: is the sketch there, how many pixels it covers, and where its box is. Today's lab is the
box, and the answer `0 0 0 0` also says whether the sketch is there.

**The file you hand in**, `submission.csv`, has one row per test photo: its `id` and its `box`. Two rows of a valid
file (the numbers are made up):

| id | box |
|---|---|
| 1300 | 81 330 160 409 |
| 1301 | 0 0 0 0 |

`sample_submission.csv` answers `0 0 0 0` for every photo.

**The score** is the IoU of your box with the true box (intersection over union: the area both boxes cover divided
by the area either box covers), averaged over all the test photos. A photo without a sketch counts 1 if you answer
`0 0 0 0` and 0 if you answer a box. The scorer cell in Part 2 computes it. Higher is better; 1 would be perfect.

**The rules.** Any library, pretrained weights allowed; no outside data. Your notebook must run from start to end.

**The starter** (next cell) puts one box in the middle of every photo. The score of its file is the **baseline**;
your points count from there.

**Today you work alone.** Nobody helps with the code and AI assistants are off. This morning's slides are yours
to use.


---
# The starter solution

Every olympiad task comes with a starter: something that hands in a valid file. Its score is the **baseline**, and your points count up from it. This task gave only the photos and a sample answer, so the starter below is ours: one box in the middle of every test photo, with no model at all. Read it before you begin; you may reuse any of its lines, the file writing above all.

```python
import pandas as pd
from PIL import Image

test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# One box in the middle of every test photo: from a quarter to three quarters of its width and of its height.
boxes = []
for i in range(len(test)):
    width, height = Image.open(os.path.join(DATA_DIR, test["image"][i])).size
    boxes.append(f"{int(width / 4)} {int(height / 4)} {int(3 * width / 4)} {int(3 * height / 4)}")

submission = pd.DataFrame({"id": test["id"], "box": boxes})
submission.to_csv("submission.csv", index=False)
print(f"Submission file saved: {len(submission)} rows")
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the data before you build anything.

## Task 1: Load the files

Read the three CSV files of `DATA_DIR` into `train`, `test` and `sample`. Print their shapes and the first rows of `train`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, sample=sample)


## Task 2: Look at photos and their boxes

Store the first four training photos in the list `first_images`, each as an array of red, green and blue values, and the number of training photos in `n_images`. Draw the four photos side by side, each with its box drawn on it. Then print anything else about the photos and the boxes that you want to know before you build a model.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_images, size=n_images)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task. Run the cell; you will use `score` all afternoon. It takes two lists of boxes written as text, the true ones first.

In [ ]:
def box_iou(true_box, pred_box):
    """IoU of two boxes written as text "x1 y1 x2 y2" (pixels). "0 0 0 0" means: no sketch."""
    a = [float(v) for v in true_box.split()]
    b = [float(v) for v in pred_box.split()]
    if a == [0, 0, 0, 0] or b == [0, 0, 0, 0]:
        if a == b:
            return 1.0          # both say: no sketch
        return 0.0              # a missed sketch, or a box where there is none
    width = min(a[2], b[2]) - max(a[0], b[0])
    height = min(a[3], b[3]) - max(a[1], b[1])
    both = max(width, 0) * max(height, 0)
    either = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - both
    return both / either


def score(true_boxes, pred_boxes):
    """The score of this task: the mean of box_iou over the photos. Higher is better; 1 is perfect."""
    total = 0
    for true_box, pred_box in zip(true_boxes, pred_boxes):
        total = total + box_iou(true_box, pred_box)
    return total / len(true_boxes)


print(round(score(["0 0 10 10", "0 0 0 0"], ["5 0 15 10", "0 0 0 0"]), 3))


## Task 3: What the scorer gives

The scorer cell above prints 0.667 for two photos: the first has a sketch, the second has none. Which sentence says what `score` gives?

- **a** For each photo, the area both boxes cover divided by the area either box covers (50 / 150 for the first); a photo where both answers are 0 0 0 0 counts 1; then the mean: (0.333 + 1) / 2 = 0.667.
- **b** For each photo, the width both boxes cover divided by the width either box covers (5 / 15 for the first); a photo where both answers are 0 0 0 0 counts 1; then the mean: (0.333 + 1) / 2 = 0.667.
- **c** For each photo with a sketch, the area both boxes cover divided by the area either box covers; photos without a sketch are left out; then the mean over the photos left: 0.333.

Store the letter in `formula`.

In [ ]:
# Your code here


## Task 4: Two answers without a model

What would two files that need no model score? `empty` answers `0 0 0 0` for every photo; `middle` is the starter's answer, one box in the middle of every photo. Score both on the boxes of `train` with `score`, and store the two numbers in the dictionary `trivial` under these two names.

In [ ]:
# Your code here


## Task 5: What the metric rewards

Photo P has no sketch; photo Q has one. File A answers a box on P and a box on Q whose IoU with the true box is 0.8. File B answers 0 0 0 0 on both photos. What do the two files score on P and Q?

- **a** A gets (0 + 0.8) / 2 = 0.4 and B gets (1 + 0) / 2 = 0.5: a right 0 0 0 0 is worth as much as a perfect box, and a box where there is no sketch is worth nothing.
- **b** A gets (0.5 + 0.8) / 2 = 0.65 and B gets (0.5 + 0) / 2 = 0.25: a box where there is no sketch still earns half, because the scorer cannot tell it is wrong.
- **c** A gets 0.8 and B gets 0: photos without a sketch are not scored, so only the box on Q counts, and an answer of 0 0 0 0 is never worth anything.

Store the letter in `rewards`.

In [ ]:
# Your code here


## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, `id_col`, the name of its id column, and `pred_col`, the name of the column of your boxes. Read all three from what you loaded; do not type them.

In [ ]:
# Your code here


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
# Your code here


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- A first file

The simplest pipeline that ends in a file comes first: aim to hand it in within the first hour. Improving comes after it.

## Task 8: Hold out validation photos

Split the ids of `train` into `train_ids` and `val_ids`, one photo in five for validation, with `random_state=0`. Print the number of photos in each part.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=train_ids, val_ids=val_ids)


## Task 9: The folder YOLO learns from

Build the folder a YOLO model trains on: the photos of `train_ids` and of `val_ids`, each with its label file, and the data YAML file that points at them. Keep in `label_rows` every label row you wrote, as a NumPy array with one row per box and four columns: the centre x, the centre y, the width and the height of the box, as YOLO reads them. Print its shape.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(label_rows)


## Task 10: The model

Create `model`, a small pretrained YOLO model, and choose the device it will train on. Store the paths of four validation photos in the list `model_batch`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, model_batch)


## Task 11: Train

Train `model` on your folder for at least 10 epochs. Then store in the list `losses` the training box loss of every epoch, in order, read from the `results.csv` file the training wrote. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.loop(losses)


## Task 12: Score the validation photos

Give every photo of `val_ids` one answer, a box or `0 0 0 0`, in the format of the file. Store in `val_score` the `score` of these answers against the true boxes of those photos: one number. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="box_iou")


## Task 13: Write your first file

Answer every test photo the same way and write `submission.csv` in the format of `sample`: its `id` and its `box`. Leave the index out. Print its shape.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
# Part 4 -- Debugging drill

A classmate wanted the area of the box of one training photo, in pixels. The cell stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it prints the area, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-typ-1")


## Task 14: Repair the classmate's cell

Change as little as you can: `drill_area` must be the area of the box of that photo, width times height.

In [ ]:
drill_row = train.iloc[0]
x1, y1, x2, y2 = drill_row["box"].split()
drill_area = (x2 - x1) * (y2 - y1)
print("area of the box in pixels:", drill_area)


In [ ]:
ck.drill_check("bug-typ-1", drill_area)


---
# Part 5 -- Challenge: your best file

Your file is on the board. Now make it better, one change at a time. A change worth keeping comes from something you found in the data or in the rules, and it raises the score of the validation photos.

## Task 15: Look again

Look again at the data and at your answers on the validation photos, all of them, for where the points are lost. Print what you find.

In [ ]:
# Your code here


## Task 16: Try a better way

Make at least one change to your pipeline. Score it on your validation photos with `score`, and print it next to `val_score`.

In [ ]:
# Your code here


## Task 17: Hand in your best file

Write `submission.csv` again with your best way. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
## Task 18: Reflect

Which change moved your validation score most, and how did you find it: in the data, in the rules or by trying? Which photos cost you the most points? If you met this task again with one hour, what would you do in the first 15 minutes?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task first**: the scorer read as code, two files without a model, what the score rewards, and the format of the file, all before any model.
- **From a table of corners to what YOLO reads**: label files, folders and a data file, written by your own code.
- **A first honest file**: a pretrained YOLO model trained, scored on validation photos with the task's own score, handed in early.
- **Look before you trust**: the validation answers examined for where the points go, and what was found used.
- **Keep a change only when the score says so**: each change scored on validation photos with the task's own score before a new file.
- **Debugging**: a type error read from the error message, located and repaired against the clock.

---

Made By **Sattam Altwaim**